This notebook provides tools to pre-generate and cache training datasets for GNN-based quantum error correction decoders.

**Benefits of caching:**
- Generate once, train many times
- ~29 min generation → ~10 sec load time
- Reproducible experiments with same data
- Can run generation overnight|

## Setup

In [1]:
import sys
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('..')

sys.path.insert(0, str(BASE_PATH))

import torch
from models import DatasetCache, visualize_sparse_graph

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Base path: {BASE_PATH}")
print(f"Datasets directory: {BASE_PATH / 'datasets'}")

C:\Users\Bill\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Base path: ..
Datasets directory: ..\datasets


## List Existing Datasets

View all cached datasets and their configurations.

In [2]:
# List all cached datasets
datasets = DatasetCache.list_datasets(base_path=BASE_PATH)

if not datasets:
    print("No cached datasets found.")
    print(f"Run the generation cells below to create some!")
else:
    print(f"Found {len(datasets)} cached dataset(s):\n")
    for ds in datasets:
        print(f"📁 {ds['name']}")
        print(f"   Samples: {ds.get('n_samples', 'unknown'):,}")
        print(f"   Distance: d={ds.get('d', '?')}")
        print(f"   Error rates: {ds.get('p_values', '?')}")
        print(f"   Size: {ds.get('size_mb', 0):.1f} MB")
        print()

No cached datasets found.
Run the generation cells below to create some!


## Generate New Datasets

Configure and generate training datasets. These will be cached to disk for fast loading during training.

In [3]:
# ============================================================
# CONFIGURATION: Define which datasets to generate
# ============================================================

# Standard error rates used across all datasets
STANDARD_P_VALUES = [0.001, 0.003, 0.005, 0.007]
STANDARD_P_WEIGHTS = [0.25, 0.25, 0.25, 0.25]  # Equal distribution

# Define datasets to generate
# Each entry: (name, distance, n_samples)
DATASETS_TO_GENERATE = [
    ("d3_baseline", 3, 1_000_000),
    ("d5_baseline", 5, 1_000_000),
    ("d7_baseline", 7, 1_000_000),
    ("d9_baseline", 9, 1_000_000),
]

# K-neighbors for graph construction
K_NEIGHBORS = 6

print("Datasets configured for generation:")
print(f"  Error rates: {STANDARD_P_VALUES}")
print(f"  Weights: {STANDARD_P_WEIGHTS}")
print(f"  K-neighbors: {K_NEIGHBORS}")
print()
for name, d, n in DATASETS_TO_GENERATE:
    print(f"  • {name}: d={d}, n={n:,}")

Datasets configured for generation:
  Error rates: [0.001, 0.003, 0.005, 0.007]
  Weights: [0.25, 0.25, 0.25, 0.25]
  K-neighbors: 6

  • d3_baseline: d=3, n=1,000,000
  • d5_baseline: d=5, n=1,000,000
  • d7_baseline: d=7, n=1,000,000
  • d9_baseline: d=9, n=1,000,000


In [4]:
# ============================================================
# GENERATE DATASETS
# ============================================================
# This cell will generate all configured datasets.
# Skip any that already exist.

datasets_dir = BASE_PATH / "datasets"

for name, d, n_samples in DATASETS_TO_GENERATE:
    # Check if dataset already exists
    if (datasets_dir / f"{name}.pt").exists():
        print(f"⏭️  Dataset '{name}' already exists, skipping...")
        continue
    
    print(f"\n{'='*60}")
    print(f"Generating dataset: {name}")
    print(f"{'='*60}")
    
    # Create and generate dataset
    cache = DatasetCache(base_path=BASE_PATH, device=device)
    cache.generate(
        d=d,
        n_samples=n_samples,
        p_values=STANDARD_P_VALUES,
        p_weights=STANDARD_P_WEIGHTS,
        k_neighbors=K_NEIGHBORS,
        verbose=True
    )
    
    # Save to disk
    cache.save(name)
    
    print(f"✅ Dataset '{name}' saved successfully!")

print(f"\n{'='*60}")
print("Generation complete!")
print(f"{'='*60}")


Generating dataset: d3_baseline

Generating dataset: d=3, n_samples=1,000,000
Error rates: [0.001, 0.003, 0.005, 0.007] (weights: [0.25, 0.25, 0.25, 0.25])
SurfaceCodeSampler initialized:
  Default error rate: 0.001
  Device: cuda
  Mode: Dynamic (supports any code distance)
SparseGraph initialized:
  K neighbors: 6
  Device: cuda
  Mode: Dynamic (supports any code distance)

Sampling 1,000,000 detection events...
Converting to PyG graphs...


Converting: 100%|██████████| 1000000/1000000 [03:16<00:00, 5097.25graph/s]



Generated 1,000,000 graphs
Dataset saved: ..\datasets\d3_baseline.pt
  Metadata: ..\datasets\d3_baseline.json
  Samples: 1,000,000
✅ Dataset 'd3_baseline' saved successfully!

Generating dataset: d5_baseline

Generating dataset: d=5, n_samples=1,000,000
Error rates: [0.001, 0.003, 0.005, 0.007] (weights: [0.25, 0.25, 0.25, 0.25])
SurfaceCodeSampler initialized:
  Default error rate: 0.001
  Device: cuda
  Mode: Dynamic (supports any code distance)
SparseGraph initialized:
  K neighbors: 6
  Device: cuda
  Mode: Dynamic (supports any code distance)

Sampling 1,000,000 detection events...
Converting to PyG graphs...


Converting: 100%|██████████| 1000000/1000000 [04:26<00:00, 3757.24graph/s]



Generated 1,000,000 graphs
Dataset saved: ..\datasets\d5_baseline.pt
  Metadata: ..\datasets\d5_baseline.json
  Samples: 1,000,000
✅ Dataset 'd5_baseline' saved successfully!

Generating dataset: d7_baseline

Generating dataset: d=7, n_samples=1,000,000
Error rates: [0.001, 0.003, 0.005, 0.007] (weights: [0.25, 0.25, 0.25, 0.25])
SurfaceCodeSampler initialized:
  Default error rate: 0.001
  Device: cuda
  Mode: Dynamic (supports any code distance)
SparseGraph initialized:
  K neighbors: 6
  Device: cuda
  Mode: Dynamic (supports any code distance)

Sampling 1,000,000 detection events...
Converting to PyG graphs...


Converting: 100%|██████████| 1000000/1000000 [05:13<00:00, 3189.18graph/s]



Generated 1,000,000 graphs
Dataset saved: ..\datasets\d7_baseline.pt
  Metadata: ..\datasets\d7_baseline.json
  Samples: 1,000,000
✅ Dataset 'd7_baseline' saved successfully!

Generating dataset: d9_baseline

Generating dataset: d=9, n_samples=1,000,000
Error rates: [0.001, 0.003, 0.005, 0.007] (weights: [0.25, 0.25, 0.25, 0.25])
SurfaceCodeSampler initialized:
  Default error rate: 0.001
  Device: cuda
  Mode: Dynamic (supports any code distance)
SparseGraph initialized:
  K neighbors: 6
  Device: cuda
  Mode: Dynamic (supports any code distance)

Sampling 1,000,000 detection events...
Converting to PyG graphs...


Converting: 100%|██████████| 1000000/1000000 [06:57<00:00, 2393.55graph/s]



Generated 1,000,000 graphs
Dataset saved: ..\datasets\d9_baseline.pt
  Metadata: ..\datasets\d9_baseline.json
  Samples: 1,000,000
✅ Dataset 'd9_baseline' saved successfully!

Generation complete!


## Load and Inspect a Dataset

Load a specific dataset to verify its contents.

In [5]:
# Load a specific dataset for inspection
DATASET_TO_INSPECT = "d5_baseline"  # Change this to inspect different datasets

try:
    cache = DatasetCache(base_path=BASE_PATH, device=device)
    cache.load(DATASET_TO_INSPECT)
    
    print(f"\nDataset: {DATASET_TO_INSPECT}")
    print(f"Total samples: {len(cache):,}")
    print(f"\nConfiguration:")
    for key, value in cache.config.items():
        print(f"  {key}: {value}")
    
    # Sample a few graphs
    graphs = cache.get_graphs(n=5)
    print(f"\nSample graph statistics:")
    for i, g in enumerate(graphs):
        print(f"  Graph {i}: {g.x.shape[0]} nodes, {g.edge_index.shape[1]} edges, label={g.y.item():.0f}")
        
except FileNotFoundError:
    print(f"Dataset '{DATASET_TO_INSPECT}' not found.")
    print("Generate it first using the cells above.")

KeyboardInterrupt: 

In [ ]:
# Visualize sample graphs from the dataset
# Find non-empty graphs (graphs with fired detectors) for visualization

NUM_GRAPHS_TO_SHOW = 3  # Number of graphs to visualize

print(f"Searching for {NUM_GRAPHS_TO_SHOW} non-empty graphs to visualize...\n")

graphs_shown = 0
for i, g in enumerate(cache.get_graphs()):
    # Only visualize graphs that have nodes (fired detectors)
    if g.x.shape[0] > 0 and graphs_shown < NUM_GRAPHS_TO_SHOW:
        print(f"{'='*60}")
        print(f"Graph {i}: {g.x.shape[0]} nodes, {g.edge_index.shape[1]} edges")
        print(f"Label: {g.y.item():.0f} (observable flip)")
        print(f"{'='*60}")
        
        visualize_sparse_graph(g, title=f"Sample {i} from {DATASET_TO_INSPECT}")
        graphs_shown += 1
    
    # Stop after showing enough graphs
    if graphs_shown >= NUM_GRAPHS_TO_SHOW:
        break

if graphs_shown == 0:
    print("No non-empty graphs found in the first samples.")

## Grow Existing Dataset

If you need more samples for an existing dataset, use `ensure_size()` to grow it.

In [ ]:
# Grow an existing dataset to a larger size
# Uncomment and modify to use

# DATASET_TO_GROW = "d3_baseline"
# TARGET_SIZE = 2_000_000  # New target size

# cache = DatasetCache(base_path=BASE_PATH, device=device)
# cache.load(DATASET_TO_GROW)
# cache.ensure_size(TARGET_SIZE)
# cache.save(DATASET_TO_GROW)  # Overwrite with larger dataset

print("Uncomment the code above to grow an existing dataset.")

## Delete a Dataset

Remove a cached dataset from disk.

In [ ]:
# Delete a dataset
# Uncomment and modify to use

# import os
# DATASET_TO_DELETE = "d3_baseline"
# 
# datasets_dir = BASE_PATH / "datasets"
# data_path = datasets_dir / f"{DATASET_TO_DELETE}.pt"
# meta_path = datasets_dir / f"{DATASET_TO_DELETE}.json"
# 
# if data_path.exists():
#     os.remove(data_path)
#     print(f"Deleted: {data_path}")
# if meta_path.exists():
#     os.remove(meta_path)
#     print(f"Deleted: {meta_path}")

print("Uncomment the code above to delete a dataset.")